# QFF 2026 Phase 1 — PS-O2 Master Guide benchmark

This notebook orchestrates the reproducible pipeline. Inputs are from page 4 of `QFF_Phase_1_Master_Guide.pdf`. The guide does not set `q`, so pass a documented participant-selected value. The notebook produces the QUBO matrix, exact classical reference, p=1 QAOA and controlled Processor A/B ideal/noisy comparisons.

## 1. Required baseline

Basis gates `{rz, sx, x, cx}`, optimization level 1, SABRE layout/routing, transpiler seed 42 and 4096 shots. The synthetic noise rates are included as a separate noisy run. The guide does not specify the readout confusion matrix; this implementation uses a symmetric 1% bit-flip assumption and labels it as such.

In [ ]:
from pathlib import Path
import json, csv, subprocess, sys
ROOT = Path.cwd()
if not (ROOT / 'main.py').exists(): raise RuntimeError('Run from the qff-o2-portfolio-phase1 repository root.')
instance = json.loads((ROOT / 'data/portfolio_instance.json').read_text())
print('Returns:', instance['expected_returns'])
print('k:', instance['selected_assets_k'])
print('q configured:', instance.get('q'))
print('Penalty A:', instance['cardinality_penalty_A'])

## 2. Run benchmark

`q=1.0` is an example participant-selected value, not a number specified by the Master Guide.

In [ ]:
Q = 1.0  # EXAMPLE ONLY: choose and justify q; Master Guide does not prescribe it.
completed = subprocess.run([sys.executable, 'main.py', '--q', str(Q)], check=True, text=True, capture_output=True)
print(completed.stdout)
if completed.stderr: print(completed.stderr)

## 3. Inspect results

The script exports the QUBO matrix, exact classical reference, QAOA parameters, ideal/noisy A/B metrics, figures and report. Objective ratio uses only feasible portfolios; valid-solution frequency is separate.

In [ ]:
with (ROOT / 'results/tables/processor_comparison.csv').open(newline='', encoding='utf-8') as f:
    for row in csv.DictReader(f): print(row)

In [ ]:
from IPython.display import display, Image
for name in ['classical_portfolio_objectives.png', 'processor_topologies.png']:
    path = ROOT / 'results/figures' / name
    if path.exists(): display(Image(filename=str(path)))